# Resilient load flow with the Knitro solver

When a network is too degraded, the **Newton-Raphson** method simply diverges: it tells you *that* there is no solution, but not *where* the problem is.

The **Knitro `RELAXED` solver** takes a different route. It restates the load flow as an optimisation problem in which the power flow equations may be violated, at a cost: each violation is carried by a *slack variable* that the objective function minimises. The solver therefore always returns an answer, and the non-zero slacks tell you exactly **where** the network is infeasible and of **which nature** the infeasibility is:

| Slack | Meaning |
|-------|---------|
| 🔴 **P** | active power imbalance at a bus |
| 🔵 **Q** | reactive power imbalance at a bus |
| 🟢 **V** | voltage set point that cannot be held |

This notebook walks through that on the IEEE 14-bus network, perturbed so that Newton-Raphson diverges:

1. run Newton-Raphson &rarr; it diverges;
2. estimate the network losses with a DC load flow;
3. run the Knitro `RELAXED` solver &rarr; it converges and exports the slacks;
4. explore the result on an interactive network area diagram.

> **Prerequisites** — Knitro installed with a valid license, `KNITRODIR` and `ARTELYS_LICENSE` set, and the custom PyPowSyBl wheel installed. See the [README](README.md).

In [ ]:
import logging
from pathlib import Path

import pandas as pd
import pypowsybl.loadflow as lf
import pypowsybl.network as pn

from slack_viz_utils import calculate_dc_losses, compute_slack_info, nad_explorer_with_slack

NETWORK_FILE = Path("data") / "ieee14-voltage-perturbation.xiidm"
EXPORT_PREFIX = Path("results") / "ieee14-perturbation"
EXPORT_PREFIX.parent.mkdir(exist_ok=True)

## 1. A network that Newton-Raphson cannot solve

The IEEE 14-bus network has been perturbed on its voltage set points. We load three independent copies of it, one per run, so that the solvers do not interfere with each other.

Newton-Raphson gives up after 16 iterations with `MAX_ITERATION_REACHED` and an active power mismatch above **1 100 000 MW** at the slack bus — a number that says nothing about which part of the network is at fault.

In [ ]:
network_nr = pn.load(str(NETWORK_FILE))  # Newton-Raphson
network_dc = pn.load(str(NETWORK_FILE))  # DC load flow, to estimate losses
network_kn = pn.load(str(NETWORK_FILE))  # Knitro RELAXED

In [ ]:
# Default Open Load Flow solver: it does not converge on this network.
lf.run_ac(network_nr, lf.Parameters(provider_parameters={"acSolverType": "NEWTON_RAPHSON"}))

## 2. Estimating the network losses

The `RELAXED` solver weights the terms of its objective function with an order of magnitude of the network active losses. A DC load flow neglects losses, but they can be re-estimated a posteriori from the DC flows — that is what `calculate_dc_losses` does. The value is passed to the solver through the `losses` parameter.

In [ ]:
lf.run_dc(network_dc)
losses = calculate_dc_losses(network_dc, network_dc.get_voltage_levels())

print(f"Estimated DC losses: {losses:.2f} MW")

## 3. Solving with Knitro `RELAXED`

Only five provider parameters are needed; every other Knitro parameter keeps its default value (see the [parameter reference](README.md#knitro-solver-parameters)).

| Parameter | Value | Why |
|-----------|-------|-----|
| `acSolverType` | `KNITRO` | use Knitro instead of Newton-Raphson |
| `solverType` | `RELAXED` | relax the equations and minimise the violations |
| `losses` | *computed above* | objective function weighting |
| `exportSolution` | path prefix | export the slacks to CSV, for the visualisation |
| `threadNumber` | `1` | deterministic, reproducible run |

Two `LoadFlowParameters` also matter here. `distributed_slack=False`, because the solver handles the imbalance itself through its P slacks. And `use_reactive_limits=False`, so that the Open Load Flow reactive limits outer loop does not move the generators before Knitro runs: the slacks then reflect the raw infeasibility of the network. That one is *optional* for `RELAXED`, but mandatory for the `USE_REACTIVE_LIMITS` formulation (see the [upstream README](https://github.com/powsybl/powsybl-open-loadflow-knitro-solver#knitro-parameters)).

In [ ]:
# INFO logs the 5 largest slacks per type; switch to DEBUG to log all of them.
logging.basicConfig()
logging.getLogger("powsybl").setLevel(logging.INFO)

In [ ]:
parameters = lf.Parameters(
    distributed_slack=False,
    use_reactive_limits=False,
    provider_parameters={
        "acSolverType": "KNITRO",
        "solverType": "RELAXED",
        "losses": str(losses),
        "exportSolution": str(EXPORT_PREFIX),
        "threadNumber": "1",
    },
)
lf.run_ac(network_kn, parameters)

## 4. Reading the exported slacks

`exportSolution` writes two CSV files:

* `<prefix>.csv` — one row per activated slack: bus, type, value in p.u., and the network elements attached to that bus;
* `<prefix>_optim_info.csv` — the solver summary: total penalty, penalty per slack type, status and iteration count.

In [ ]:
slack_export = pd.read_csv(f"{EXPORT_PREFIX}.csv", sep=";")
slack_export

In [ ]:
pd.read_csv(f"{EXPORT_PREFIX}_optim_info.csv", sep=";")

## 5. Interactive visualisation

The export only lists the buses that carry a slack. `compute_slack_info` joins it onto **every** bus of the network, so that the explorer can tell "no slack here" apart from "bus missing from the export".

The solver exports one block of rows per outer loop iteration; `outerloop=0` selects the first one.

In [ ]:
slack_info = compute_slack_info(network_kn, slack_export, outerloop=0)
slack_info[["id", "bus_id", "voltage_level_id", "type", "slackValue_pu", "has_slack"]]

Select one or more voltage levels on the left to display them on the diagram and read the details of their slacks. The **Slack info** dropdown narrows the list down to the voltage levels carrying a given slack type.

In [ ]:
nad_explorer_with_slack(network_kn, slack_info)